# Facility Water Treatment Energy Estimation

**AI Services MS2 | Supplementary Practice C | 60 Marks | Exactly 20 Evaluation Checks**

**Instructor reference notebook — complete working code.**

**Business task.** A water-treatment facility runs pumps and chemical dosing systems throughout the day. Each operational reading records water flow, pump operating hours, pressure, ambient temperature, occupancy and chemical usage. The company wants to estimate electrical energy use for new operating intervals.

**Objective.** Build a numeric-only regression notebook. Convert noisy sensor readings safely, engineer time and flow-rate features, use median imputation followed by log1p and scaling, compare Ridge Regression and Random Forest, and select by the smallest held-back MAE.

## 1. Imports, parameters and feature groups

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "energy_kwh"

RAW_NUMERIC = ["flow_m3", "occupancy", "pump_hours", "ambient_temp_c", "chemical_litres", "pressure_bar"]

FEATURES = RAW_NUMERIC + ["reading_hour", "reading_month", "is_weekend", "flow_per_pump_hour"]

## Implementation contract — independently testable functions

**1. `load_data(filename: str) -> pd.DataFrame`**  
**Returns:** Return cleaned DataFrame with record ID, optional target and ten numeric feature columns.  
• Read CSV, keep first reading_id, parse reading_ts from mixed formats and remove invalid timestamps.  
• Coerce six raw measurements to numeric and convert +inf/-inf and negative values to NaN. Negative numerical features are intentionally treated as missing for compatibility with log1p.  
• Create reading_hour, reading_month and is_weekend=(dayofweek >= 5). Calculate flow_per_pump_hour with pump_hours > 0; invalid ratios remain NaN.

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    """Clean numeric sensor readings, mixed timestamps, and engineer four new numerical fields."""

    df = pd.read_csv(filename, dtype={"reading_id": "string"})
    df = df.drop_duplicates("reading_id", keep="first").copy()
    df["reading_ts"] = pd.to_datetime(df["reading_ts"], errors="coerce", dayfirst=True, format="mixed")
    df = df.dropna(subset=["reading_ts"]).copy()
    
    for col in RAW_NUMERIC:
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].where(df[col] >= 0)  # nonnegative predictors for log1p
    df["reading_hour"] = df["reading_ts"].dt.hour
    df["reading_month"] = df["reading_ts"].dt.month
    df["is_weekend"] = (df["reading_ts"].dt.dayofweek >= 5).astype(int)
    df["flow_per_pump_hour"] = (
        df["flow_m3"] / df["pump_hours"].where(df["pump_hours"] > 0)).replace([np.inf, -np.inf], np.nan)
    
    return df.reset_index(drop=True)

**2. `build_pipeline(algorithm) -> Pipeline`**  
**Returns:** Return an unfitted four-step numeric Scikit-learn Pipeline.  
• Use the exact sequence: SimpleImputer(strategy="median"), FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler(), and the given model.  
• Unlike the other assessments, this pipeline is numeric-only; do not add a categorical encoder or ColumnTransformer.

In [ ]:
def build_pipeline(algorithm) -> Pipeline:
    """Numeric-only Pipeline: median imputer, log1p transformer, scaler and estimator."""
    
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", StandardScaler()),
        ("model", algorithm)
    ])

**3. `prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]`**  
**Returns:** Return X_train, X_test, y_train, y_test.  
• Use the common loader on both labeled files; coerce energy_kwh to numeric, replace infinities, and remove rows with missing or negative target values.  
• Return only the ten predictor columns in the X objects, with aligned numeric target Series.

In [ ]:
def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Filter invalid, missing, infinite or negative energy targets and return supervised splits."""

    train_df, test_df = load_data(train_file), load_data(test_file)
    
    for df in (train_df, test_df):
        df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").replace([np.inf, -np.inf], np.nan)
        df.drop(df.index[df[TARGET].isna() | (df[TARGET] < 0)], inplace=True)
    
    return train_df[FEATURES], test_df[FEATURES], train_df[TARGET], test_df[TARGET]

**4. `train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]`**  
**Returns:** Return fitted ridge_regression and random_forest pipelines.  
• Ridge(alpha=1.0) and RandomForestRegressor(n_estimators=80, max_depth=10, min_samples_leaf=2, random_state=42, n_jobs=-1).  
• Fit two distinct complete pipelines, with fitted preprocessing retained in each.

In [ ]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    """Train Ridge and RandomForestRegressor with complete independent pipelines."""

    models = {
        "ridge_regression": Ridge(alpha=1.0),
        "random_forest": RandomForestRegressor(n_estimators=80, max_depth=10,
                    min_samples_leaf=2, random_state=42, n_jobs=-1)
    }
    trained_models = {}
    
    for name, algorithm in models.items():
        pipeline = build_pipeline(algorithm)
        pipeline.fit(X_train, y_train)
        trained_models[name] = pipeline
    
    return trained_models

**5. `evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame`**  
**Returns:** Return a sorted two-row comparison DataFrame.  
• Calculate MAE, RMSE and R2 using held-back labels. Exact columns: model_name, mae, rmse, r2.  
• Sort by MAE ascending, then RMSE ascending, then model_name ascending and reset the index. Do not refit.

In [ ]:
def evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame,y_test: pd.Series) -> pd.DataFrame:
    """Compare regression models by MAE ascending, then RMSE ascending."""

    results = []
    
    for name, pipeline in trained_models.items():
        predictions = pipeline.predict(X_test)
        results.append({"model_name": name,
                        "mae": mean_absolute_error(y_test, predictions),
                        "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
                        "r2": r2_score(y_test, predictions)})
    
    return pd.DataFrame(results).sort_values(["mae", "rmse", "model_name"], ascending=[True, True, True]).reset_index(drop=True)

**6. `select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]`**  
**Returns:** Return best_name and the corresponding already-fitted pipeline.  
• Take the first row of the MAE-ranked comparison.

In [ ]:
def select_best_model(comparison: pd.DataFrame,trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    """Return first model in MAE-ranked comparison and existing fitted pipeline."""

    name = str(comparison.iloc[0]["model_name"])
    
    return name, trained_models[name]

**7. `predict_energy(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame`**  
**Returns:** Return reading_id, predicted_energy_kwh.  
• Predict using only the ten numeric features. Clip negative predicted values to zero and round to two decimal places; preserve input record order.

In [ ]:
def predict_energy(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    """Predict nonnegative treatment energy and round to two decimals."""

    values = np.maximum(model.predict(df[FEATURES]), 0).round(2)
    
    return pd.DataFrame({"reading_id": df["reading_id"].to_numpy(),"predicted_energy_kwh": values})

## Training and saved-model prediction

The training and prediction workflows are intentionally separate. Training fits and persists the chosen estimator with its preprocessing. Prediction must load the saved model and process new rows without calling `fit()`.

**`train() -> tuple[str, pd.DataFrame]`** — Return chosen model name and comparison report. Run fitting/evaluation/selection and save best_model.pkl and model_comparison.csv.

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    """Train/evaluate and write the selected persisted pipeline and comparison."""

    X_train, X_test, y_train, y_test = prepare_datasets("water_train.csv", "water_test.csv")
    trained = train_models(X_train, y_train)
    comparison = evaluate_models(trained, X_test, y_test)
    name, model = select_best_model(comparison, trained)
    joblib.dump(model, "best_model.pkl")
    comparison.to_csv("model_comparison.csv", index=False)
    
    return name, comparison

**`predict() -> pd.DataFrame`** — Return predictions for queued readings. Load water_predict.csv and existing best_model.pkl; write water_predictions.csv without calling fit.

**Required output files:** `best_model.pkl`, `model_comparison.csv`, `water_predictions.csv`

In [ ]:
def predict() -> pd.DataFrame:
    """Independently reload best_model.pkl and write water_predictions.csv."""

    df = load_data("water_predict.csv")
    result = predict_energy(joblib.load("best_model.pkl"), df)
    result.to_csv("water_predictions.csv", index=False)
    
    return result

## 4. Run the workflow

Execute the **training** call first, then the **prediction** call. You can also run these in separate notebook sessions after a model has been saved.

In [ ]:
best_name, comparison = train()
comparison

In [ ]:
result = predict()
result.head()